# Execução dos SLMs sem RAG — versão enxuta

Notebook exclusivo para executar **TinyLlama, Qwen2.5, Mistral e Zephyr** sem recuperação de contexto.

O objetivo desta etapa é somente:

- gerar os relatórios técnicos sem RAG;
- preservar os mesmos casos, parâmetros e configurações dos experimentos originais;
- registrar latência, VRAM, tokens, validade e truncamento;
- exportar um CSV padronizado para a análise posterior RAG vs. No-RAG.

As métricas semânticas serão calculadas posteriormente, utilizando os resultados originais com RAG e os relatórios gerados por este notebook.

## 1. Instalação

In [ ]:
# ============================================================
# INSTALAÇÃO FIXADA — SOMENTE INFERÊNCIA
# ============================================================
!pip install -q \
  pandas==2.2.2 \
  transformers==4.45.2 \
  accelerate==0.34.2 \
  bitsandbytes==0.49.2 \
  sentencepiece \
  huggingface_hub \
  tqdm

In [ ]:
# Execute uma vez após a instalação para reiniciar o runtime.
import os
os.kill(os.getpid(), 9)

## 2. Ambiente e autenticação

In [ ]:
# ============================================================
# HUGGING FACE — CONFIGURAÇÃO DE DOWNLOAD
# Execute antes de importar huggingface_hub, transformers
# ou sentence_transformers.
# ============================================================

import os

# Desativa o backend Xet, que apresentou URLs assinadas inválidas.
os.environ["HF_HUB_DISABLE_XET"] = "1"

# Aumenta os tempos de espera para arquivos grandes.
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "300"
os.environ["HF_HUB_ETAG_TIMEOUT"] = "60"

# Mantém a saída limpa.
os.environ["HF_HUB_VERBOSITY"] = "warning"

print("Configuração de download do Hugging Face aplicada.")
print("HF_HUB_DISABLE_XET =", os.environ["HF_HUB_DISABLE_XET"])


In [ ]:
# ============================================================
# LOGIN SEGURO NO HUGGING FACE
# O Secret do Colab deve se chamar exatamente HF_token.
# ============================================================

from google.colab import userdata
from huggingface_hub import login, whoami
import os

hf_token = userdata.get("HF_TOKEN")

if not hf_token:
    raise RuntimeError(
        "HF_token não encontrado nos Secrets do Colab. "
        "Habilite o acesso deste notebook ao Secret."
    )

# Nome padrão reconhecido internamente pelas bibliotecas do Hugging Face.
os.environ["HF_TOKEN"] = hf_token

login(
    token=hf_token,
    add_to_git_credential=False
)

hf_account = whoami(token=hf_token)

print("Login no Hugging Face realizado com sucesso.")
print("Usuário autenticado:", hf_account.get("name", "não identificado"))


In [ ]:
from pathlib import Path
import os
import json
import time
import gc
import textwrap
import hashlib
import platform
from typing import Dict, List, Any

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    set_seed,
)

print("torch:", torch.__version__)
print("CUDA disponível:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 3. Configuração

In [ ]:
RUN_MODELS = ["tinyllama", "qwen", "mistral", "zephyr"]

GLOBAL_CONFIG = {
    "experiment_version": "without_rag_inference_all_models_v2",
    "cnn_predictions_path": "/content/holdout_test_predictions.csv",
    "results_dir": "/content/results_without_rag",

    # Mesmos parâmetros usados nos experimentos originais
    "max_input_tokens": 1280,
    "max_new_tokens": 700,
    "do_sample": True,
    "temperature": 0.2,
    "top_p": 0.9,
    "top_k": None,
    "repetition_penalty": 1.1,
    "base_seed": 42,

    "minimum_report_chars": 120,
    "fail_on_truncation": True,
}

MODEL_PROFILES = {
    "mistral": {
        "model_name": "mistralai/Mistral-7B-Instruct-v0.2",
        "model_label": "Mistral-7B-Instruct-v0.2",
        "loading_mode": "4bit_nf4",
        "load_in_4bit": True,
        "bnb_4bit_quant_type": "nf4",
        "bnb_4bit_compute_dtype": "float16",
        "bnb_4bit_use_double_quant": False,
        "torch_dtype": "float16",
        "trust_remote_code": False,
        "use_fast_tokenizer": True,
        "padding_side": None,
        "invalid_rules": "mistral",
    },
    "qwen": {
        "model_name": "Qwen/Qwen2.5-3B-Instruct",
        "model_label": "Qwen2.5-3B-Instruct-FP16",
        "loading_mode": "fp16",
        "load_in_4bit": False,
        "torch_dtype": "float16",
        "trust_remote_code": True,
        "use_fast_tokenizer": True,
        "padding_side": None,
        "invalid_rules": "qwen",
    },
    "tinyllama": {
        "model_name": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
        "model_label": "TinyLlama-1.1B-Chat-v1.0-FP16",
        "loading_mode": "fp16",
        "load_in_4bit": False,
        "torch_dtype": "float16",
        "trust_remote_code": False,
        "use_fast_tokenizer": True,
        "padding_side": "left",
        "invalid_rules": "tinyllama",
    },
    "zephyr": {
        "model_name": "HuggingFaceH4/zephyr-7b-beta",
        "model_label": "Zephyr-7B-Beta-4bit",
        "loading_mode": "4bit_nf4",
        "load_in_4bit": True,
        "bnb_4bit_quant_type": "nf4",
        "bnb_4bit_compute_dtype": "float16",
        "bnb_4bit_use_double_quant": True,
        "torch_dtype": "float16",
        "trust_remote_code": False,
        "use_fast_tokenizer": True,
        "padding_side": None,
        "invalid_rules": "generic",
    },
}

results_dir = Path(GLOBAL_CONFIG["results_dir"])
results_dir.mkdir(parents=True, exist_ok=True)
print("Modelos:", RUN_MODELS)
print("Condição: without_rag")
print("Saída:", results_dir)

## 4. Classes e casos da CNN

In [ ]:
CLASS_ALIASES = {
    "lack of phase": "Phase Loss",
    "phase loss": "Phase Loss",
    "normal": "Normal Operation",
    "normal operation": "Normal Operation",
    "bearing failure": "Bearing Failure",
    "blocked rotor": "Blocked Rotor",
    "overheating": "Overheating",
    "ventilation defect": "Ventilation Defect",
}

def canonicalize_class(name: str) -> str:
    key = str(name).strip().lower()
    return CLASS_ALIASES.get(key, str(name).strip())

In [ ]:
# ============================================================
# UPLOAD DO CSV DE SAÍDA DA CNN (HOLD-OUT)
# ============================================================

from google.colab import files
import pandas as pd
from pathlib import Path

print("Selecione o arquivo holdout_test_predictions.csv")

uploaded = files.upload()

if len(uploaded) == 0:
    raise RuntimeError("Nenhum arquivo foi enviado.")

csv_name = list(uploaded.keys())[0]

# Caminho que será utilizado em todo o notebook
CNN_PREDICTIONS_PATH = Path("/content/holdout_test_predictions.csv")

# Renomeia automaticamente para manter o restante do código igual
Path(csv_name).rename(CNN_PREDICTIONS_PATH)

print(f"Arquivo salvo em: {CNN_PREDICTIONS_PATH}")

# Carrega para conferência
holdout_df = pd.read_csv(CNN_PREDICTIONS_PATH)

print("\nArquivo carregado com sucesso!")
print(f"Número de amostras: {len(holdout_df)}")
print(f"Número de colunas: {len(holdout_df.columns)}")

display(holdout_df.head())

In [ ]:
# ============================================================
# CASOS REAIS DA CNN — SELEÇÃO AUTOMÁTICA NO HOLD-OUT
# Uma amostra corretamente classificada por classe, cuja confiança
# seja a mais próxima da mediana da própria classe.
# ============================================================
CNN_CSV_PATH = GLOBAL_CONFIG["cnn_predictions_path"]

if not Path(CNN_CSV_PATH).exists():
    raise FileNotFoundError(
        f"Arquivo da CNN não encontrado: {CNN_CSV_PATH}. "
        "Envie holdout_test_predictions.csv para o Colab."
    )

holdout_df = pd.read_csv(CNN_CSV_PATH)

COLUMN_CANDIDATES = {
    "true_class": ["true_class", "true_label", "actual_class", "target_class"],
    "predicted_class": [
        "predicted_class", "pred_class", "prediction", "predicted_label"
    ],
    "confidence": [
        "predicted_confidence", "confidence", "max_probability", "probability"
    ],
    "sample_id": [
        "sample_id", "image_id", "filename", "file_name", "path", "index"
    ],
}

def resolve_column(df, candidates, required=True):
    for name in candidates:
        if name in df.columns:
            return name
    if required:
        raise KeyError(
            f"Nenhuma das colunas esperadas foi encontrada: {candidates}. "
            f"Colunas disponíveis: {list(df.columns)}"
        )
    return None

true_col = resolve_column(holdout_df, COLUMN_CANDIDATES["true_class"])
pred_col = resolve_column(holdout_df, COLUMN_CANDIDATES["predicted_class"])
conf_col = resolve_column(holdout_df, COLUMN_CANDIDATES["confidence"])
id_col = resolve_column(
    holdout_df,
    COLUMN_CANDIDATES["sample_id"],
    required=False
)

working_df = holdout_df.copy()
working_df["_true_canonical"] = working_df[true_col].map(canonicalize_class)
working_df["_pred_canonical"] = working_df[pred_col].map(canonicalize_class)
working_df["_confidence"] = pd.to_numeric(
    working_df[conf_col],
    errors="coerce"
)

working_df = working_df[
    working_df["_true_canonical"] == working_df["_pred_canonical"]
].dropna(subset=["_confidence"])

selected_rows = []

for class_name, group in working_df.groupby("_true_canonical", sort=True):
    median_confidence = group["_confidence"].median()
    chosen = (
        group.assign(
            _distance_to_median=(
                group["_confidence"] - median_confidence
            ).abs()
        )
        .sort_values(
            ["_distance_to_median", "_confidence"],
            ascending=[True, True]
        )
        .iloc[0]
    )
    selected_rows.append(chosen)

selected_cases_df = pd.DataFrame(selected_rows).reset_index(drop=True)

test_cases = []
for row_number, row in selected_cases_df.iterrows():
    raw_id = (
        str(row[id_col])
        if id_col is not None
        else f"holdout_row_{int(row.name)}"
    )

    test_cases.append({
        "sample_id": raw_id,
        "true_class": row["_true_canonical"],
        "predicted_class": row["_pred_canonical"],
        "confidence": float(row["_confidence"]),

        # A CNN classifica a imagem, mas não gera descrição textual da termografia.
        # Mantemos vazio para não introduzir evidência manual.
        "observed_evidence": "",

        # Rastreabilidade
        "cnn_source_row": int(row.name),
        "selection_rule": "correct_prediction_closest_to_class_median_confidence",
    })

expected_classes = set(working_df["_true_canonical"].unique())
selected_classes = {case["predicted_class"] for case in test_cases}

if selected_classes != expected_classes:
    raise RuntimeError(
        "A seleção não representou todas as classes corretamente classificadas. "
        f"Esperadas: {sorted(expected_classes)}; "
        f"selecionadas: {sorted(selected_classes)}"
    )

print(
    f"Hold-out: {len(holdout_df)} amostras | "
    f"corretas: {len(working_df)} | "
    f"casos selecionados: {len(test_cases)}"
)
display(pd.DataFrame(test_cases))


## 6. Carregamento sequencial dos modelos

In [ ]:
# ============================================================
# ADAPTADOR ÚNICO — CARREGAMENTO DIRETO DO SLM
# Não utiliza snapshot_download nem pasta local manual.
# ============================================================

def dtype_from_string(dtype_name: str):
    """Converte o dtype informado na configuração para torch.dtype."""

    if dtype_name == "bfloat16":
        return torch.bfloat16

    if dtype_name == "float32":
        return torch.float32

    return torch.float16


def clear_cuda():
    """Libera memória antes do carregamento do modelo."""

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def load_slm(
    config: Dict[str, Any],
    hf_token: str
):
    clear_cuda()

    if not hf_token:
        raise RuntimeError(
            "HF_TOKEN não está disponível para carregar o modelo."
        )

    model_name = config["model_name"]

    torch_dtype = dtype_from_string(
        config.get("torch_dtype", "float16")
    )

    print("=" * 70)
    print("Carregando modelo:", model_name)
    print("Modo:", config.get("loading_mode"))
    print("=" * 70)

    # ========================================================
    # NÃO USAMOS MAIS SNAPSHOT LOCAL DO MODELO
    # ========================================================

    # model_slug = model_name.split("/")[-1]
    # local_model_dir = f"/content/models/{model_slug}"

    # model_snapshot_path = snapshot_download(
    #     repo_id=model_name,
    #     local_dir=local_model_dir,
    #     token=hf_token,
    # )

    # O snapshot completo poderia baixar arquivos desnecessários
    # ou depender de várias transferências paralelas.
    # Por isso o modelo volta a ser carregado diretamente pelo
    # identificador oficial do Hugging Face.

    # ========================================================
    # TOKENIZER
    # ========================================================

    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        use_fast=config.get(
            "use_fast_tokenizer",
            True
        ),
        trust_remote_code=config.get(
            "trust_remote_code",
            False
        ),

        # Token passado explicitamente.
        token=hf_token,

        # Não usar local_files_only=True porque o modelo
        # pode ainda não estar integralmente no cache.
        local_files_only=False,
    )

    if tokenizer.pad_token is None:
        if tokenizer.eos_token is None:
            raise RuntimeError(
                "O tokenizer não possui pad_token nem eos_token."
            )

        tokenizer.pad_token = tokenizer.eos_token

    if config.get("padding_side") is not None:
        tokenizer.padding_side = config["padding_side"]

    # Configuração específica do Qwen.
    # Não interfere no Zephyr, Mistral ou TinyLlama.
    if config.get("model_key") == "qwen":
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id

    # ========================================================
    # ARGUMENTOS GERAIS DO MODELO
    # ========================================================

    model_kwargs = {
        "device_map": "auto",
        "torch_dtype": torch_dtype,
        "trust_remote_code": config.get(
            "trust_remote_code",
            False
        ),

        # Token explicitamente propagado ao download dos pesos.
        "token": hf_token,

        "low_cpu_mem_usage": True,

        # Não usar local_files_only=True nesta versão.
        "local_files_only": False,
    }

    # ========================================================
    # QUANTIZAÇÃO OPCIONAL EM 4 BITS
    # Usada por Zephyr e Mistral.
    # ========================================================

    if config.get("load_in_4bit", False):

        compute_dtype = dtype_from_string(
            config.get(
                "bnb_4bit_compute_dtype",
                "float16"
            )
        )

        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,

            bnb_4bit_quant_type=config.get(
                "bnb_4bit_quant_type",
                "nf4"
            ),

            bnb_4bit_compute_dtype=compute_dtype,

            bnb_4bit_use_double_quant=config.get(
                "bnb_4bit_use_double_quant",
                False
            ),
        )

        model_kwargs["quantization_config"] = bnb_config

        print("Quantização: 4-bit")
        print(
            "Tipo:",
            config.get(
                "bnb_4bit_quant_type",
                "nf4"
            )
        )
        print(
            "Double quant:",
            config.get(
                "bnb_4bit_use_double_quant",
                False
            )
        )

    else:
        print("Quantização: não utilizada")

    # ========================================================
    # CARREGAMENTO DIRETO DO MODELO
    # ========================================================

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        **model_kwargs
    )

    model.eval()

    # ========================================================
    # CONFIGURAÇÕES DE GERAÇÃO
    # ========================================================

    model.config.pad_token_id = tokenizer.pad_token_id

    model.generation_config.pad_token_id = (
        tokenizer.pad_token_id
    )

    model.generation_config.eos_token_id = (
        tokenizer.eos_token_id
    )

    model.config.use_cache = True

    # ========================================================
    # VALIDAÇÃO
    # ========================================================

    print("\nModelo carregado com sucesso:", model_name)
    print("Label:", config.get("model_label"))
    print("Modo:", config.get("loading_mode"))
    print("Observação:", config.get("notes"))
    print("Tokenizer:", tokenizer.name_or_path)
    print("Model:", model.config._name_or_path)
    print(
        "EOS:",
        tokenizer.eos_token,
        tokenizer.eos_token_id
    )
    print(
        "PAD:",
        tokenizer.pad_token,
        tokenizer.pad_token_id
    )
    print(
        "Chat template existe?",
        tokenizer.chat_template is not None
    )

    if torch.cuda.is_available():
        torch.cuda.synchronize()

        print(
            "VRAM alocada:",
            round(
                torch.cuda.memory_allocated() / 1024**3,
                3
            ),
            "GB"
        )

        print(
            "VRAM reservada:",
            round(
                torch.cuda.memory_reserved() / 1024**3,
                3
            ),
            "GB"
        )

    return tokenizer, model


## 7. Prompt sem RAG

In [ ]:
def build_without_rag_prompt(
    predicted_class: str,
    confidence: float,
    observed_evidence: str = "",
) -> str:
    canonical_class = canonicalize_class(predicted_class)
    evidence_text = (
        str(observed_evidence).strip()
        or "No textual thermal evidence was produced by the CNN; only the predicted class and confidence are available."
    )

    return textwrap.dedent(f"""
    You are an industrial maintenance specialist in induction motor fault diagnosis using infrared thermography.

    CNN prediction:
    - Predicted fault: {canonical_class}
    - Confidence: {confidence:.3f}

    Observed evidence:
    {evidence_text}

    No external technical context is available in this condition.

    Generate ONE concise technical diagnostic report in English using your internal parametric knowledge.

    The report MUST contain EXACTLY the following five sections:

    1. Technical diagnosis
    2. Supporting evidence
    3. Probable causes
    4. Recommended maintenance actions
    5. Severity and risk assessment

    Mandatory rules:

    - Use the CNN prediction as the confirmed diagnostic input.
    - Clearly distinguish confirmed information from probable causes.
    - Never invent measurements, temperatures, standards, inspections, citations, or observed evidence.
    - Do not claim access to retrieved documents or external technical knowledge.
    - Do not state that the CNN observed textual thermal patterns unless explicitly provided.
    - Write exactly five sections using ONLY the headings above.
    - Use no more than THREE short sentences per section.
    - Keep the entire report between approximately 180 and 300 words.
    - Do NOT add introductions, conclusions, notes, bullet lists, or extra sections.
    - Do NOT repeat information across sections.
    - Finish immediately after Section 5 with a complete sentence.

    Produce only the report.
    """).strip()

## 8. Geração e validação

In [ ]:
def is_invalid_report(
    report: str,
    profile: str,
    minimum_chars: int = 120
) -> bool:

    text = report.strip()
    low = text.lower()

    forbidden_placeholders = [
        "[insert",
        "insert date",
        "insert location",
        "insert equipment",
        "insert temperature",
        "[date]",
        "[location]",
        "[equipment",
    ]

    generic_invalid = (
        len(text) < minimum_chars
        or text in ["", "</s>", "<s>"]
        or "traceback" in low
        or "cuda" in low
        or any(item in low for item in forbidden_placeholders)
    )

    if generic_invalid:
        return True

    if profile == "mistral":
        return "<unk>" in text or "ACHE" in text

    if profile == "qwen":
        return text.count("!") > 20

    if profile == "tinyllama":
        return (
            text.count("!") > 10
            or text.count("#") > 10
            or len(set(text.replace(" ", ""))) < 10
            or "- - -" in text
            or text.count("\n      -") > 20
            or text.count("\n        -") > 20
            or text.count("Fault Type:") > 3
            or text.count("Risk Level:") > 3
            or "technical maintenance report for" in low
            or "\ndate:" in low
            or "\nlocation:" in low
        )

    return False
# def is_invalid_report(report: str, profile: str, minimum_chars: int = 120) -> bool:
#     text = report.strip()
#     low = text.lower()

#     generic_invalid = (
#         len(text) < minimum_chars
#         or text in ["", "</s>", "<s>"]
#         or "Traceback" in text
#         or "CUDA" in text
#     )

#     if generic_invalid:
#         return True

#     if profile == "mistral":
#         return "<unk>" in text or "ACHE" in text

#     if profile == "qwen":
#         return text.count("!") > 20

#     if profile == "tinyllama":
#         return (
#             text.count("!") > 10
#             or text.count("#") > 10
#             or len(set(text.replace(" ", ""))) < 10
#             or "- - -" in text
#             or text.count("\n      -") > 20
#             or text.count("\n        -") > 20
#             or text.count("Fault Type:") > 3
#             or text.count("Risk Level:") > 3
#         )

#     return False


def apply_model_chat_template(prompt: str) -> str:
    messages = [{"role": "user", "content": prompt.strip()}]

    if hasattr(tokenizer, "apply_chat_template") and tokenizer.chat_template is not None:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

    return prompt.strip()


def configure_seed(seed: int) -> None:
    set_seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def generate_report(
    prompt: str,
    config: Dict[str, Any],
    seed: int
) -> Dict[str, Any]:

    configure_seed(seed)
    input_text = apply_model_chat_template(prompt)

    raw_inputs = tokenizer(
        input_text,
        return_tensors="pt",
        padding=False,
        truncation=False,
    )

    raw_input_tokens = int(raw_inputs["input_ids"].shape[-1])
    max_input_tokens = int(config["max_input_tokens"])
    input_was_truncated = raw_input_tokens > max_input_tokens

    if input_was_truncated:
        raise RuntimeError(
            f"Prompt excedeu o limite final: "
            f"{raw_input_tokens} > {max_input_tokens} tokens. "
            "Nenhuma geração foi executada."
        )

    inputs = {
        key: value.to(model.device)
        for key, value in raw_inputs.items()
    }

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()

    generation_kwargs = {
        "input_ids": inputs["input_ids"],
        "attention_mask": inputs.get("attention_mask"),
        "max_new_tokens": int(config["max_new_tokens"]),
        "do_sample": bool(config["do_sample"]),
        "repetition_penalty": float(config["repetition_penalty"]),
        "pad_token_id": (
            tokenizer.pad_token_id
            if tokenizer.pad_token_id is not None
            else tokenizer.eos_token_id
        ),
        "eos_token_id": tokenizer.eos_token_id,
        "use_cache": True,
        "return_dict_in_generate": True,
    }

    if generation_kwargs["do_sample"]:
        if config.get("temperature") is not None:
            generation_kwargs["temperature"] = float(config["temperature"])
        if config.get("top_p") is not None:
            generation_kwargs["top_p"] = float(config["top_p"])
        if config.get("top_k") is not None:
            generation_kwargs["top_k"] = int(config["top_k"])

    generation_kwargs = {
        key: value
        for key, value in generation_kwargs.items()
        if value is not None
    }

    start = time.perf_counter()
    with torch.inference_mode():
        generation_output = model.generate(**generation_kwargs)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    latency = time.perf_counter() - start

    sequences = generation_output.sequences
    generated_ids = sequences[0][raw_input_tokens:]

    output_tokens = int(generated_ids.shape[-1])
    eos_id = tokenizer.eos_token_id
    ended_with_eos = bool(
        eos_id is not None
        and (generated_ids == eos_id).any().item()
    )

    reached_token_limit = (
        output_tokens >= int(config["max_new_tokens"])
    )
    was_truncated = bool(
        reached_token_limit and not ended_with_eos
    )

    if ended_with_eos:
        finish_reason = "eos_token"
    elif reached_token_limit:
        finish_reason = "length"
    else:
        finish_reason = "other"

    report = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()

    peak_vram_gb = None
    if torch.cuda.is_available():
        peak_vram_gb = round(
            torch.cuda.max_memory_allocated() / 1024**3,
            3
        )

    valid_report = (
        not is_invalid_report(
            report,
            config.get("invalid_rules", "generic"),
            minimum_chars=int(
                config.get("minimum_report_chars", 120)
            ),
        )
        and not was_truncated
        and not input_was_truncated
    )

    return {
        "report": report,
        "valid_report": valid_report,
        "latency_seconds": latency,
        "input_tokens": raw_input_tokens,
        "raw_input_tokens": raw_input_tokens,
        "output_tokens": output_tokens,
        "peak_vram_gb": peak_vram_gb,
        "seed": seed,
        "finish_reason": finish_reason,
        "ended_with_eos": ended_with_eos,
        "reached_token_limit": reached_token_limit,
        "was_truncated": was_truncated,
        "input_was_truncated": input_was_truncated,
    }


## 9. Execução dos quatro modelos — somente sem RAG

In [ ]:
def stable_hash(text: str) -> str:
    return hashlib.sha256(str(text).encode("utf-8")).hexdigest()


def unload_current_model():
    global model, tokenizer
    try:
        del model
    except Exception:
        pass
    try:
        del tokenizer
    except Exception:
        pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


def run_without_rag(cases: List[Dict[str, Any]], config: Dict[str, Any]) -> pd.DataFrame:
    rows = []

    for case_index, case in enumerate(
        tqdm(cases, desc=f"{config['model_label']} | without_rag")
    ):
        seed = int(config["base_seed"]) + case_index
        canonical_class = canonicalize_class(case["predicted_class"])

        prompt = build_without_rag_prompt(
            predicted_class=canonical_class,
            confidence=float(case["confidence"]),
            observed_evidence=case.get("observed_evidence", ""),
        )

        generated = generate_report(prompt, config, seed)

        rows.append({
            **case,
            "predicted_class": canonical_class,
            "model_key": config["model_key"],
            "model_name": config["model_name"],
            "model_label": config["model_label"],
            "loading_mode": config["loading_mode"],
            "condition": "without_rag",
            "use_rag": False,
            "prompt": prompt,
            "prompt_sha256": stable_hash(prompt),
            **generated,
        })

    return pd.DataFrame(rows)


all_runs = []

for model_key in RUN_MODELS:
    print("\n" + "=" * 90)
    print("CARREGANDO:", model_key)
    print("=" * 90)

    EXPERIMENT_CONFIG = {
        **GLOBAL_CONFIG,
        **MODEL_PROFILES[model_key],
        "model_key": model_key,
    }

    try:
        tokenizer, model = load_slm(EXPERIMENT_CONFIG, hf_token=hf_token)
        run_df = run_without_rag(test_cases, EXPERIMENT_CONFIG)
        all_runs.append(run_df)

        partial_path = results_dir / f"generation_{model_key}_without_rag.csv"
        run_df.to_csv(partial_path, index=False, encoding="utf-8")
        print("Salvo:", partial_path)

    except Exception as error:
        error_path = results_dir / f"error_{model_key}_without_rag.txt"
        error_path.write_text(
            f"{type(error).__name__}: {error}",
            encoding="utf-8",
        )
        print(f"Falha em {model_key}: {error}")
        print("Erro registrado em:", error_path)

    finally:
        unload_current_model()

if not all_runs:
    raise RuntimeError("Nenhum modelo concluiu a execução.")

results_df = pd.concat(all_runs, ignore_index=True)
combined_path = results_dir / "all_generations_without_rag.csv"
results_df.to_csv(combined_path, index=False, encoding="utf-8")

print("\nGerações consolidadas:", combined_path)
display(results_df[[
    "model_key", "sample_id", "predicted_class", "valid_report",
    "latency_seconds", "input_tokens", "output_tokens", "peak_vram_gb"
]])

## Resumo operacional por modelo


In [ ]:
operational_summary = (
    results_df
    .groupby("model_key", as_index=False)
    .agg(
        n_cases=("sample_id", "count"),
        latency_mean_s=("latency_seconds", "mean"),
        latency_std_s=("latency_seconds", "std"),
        peak_vram_gb=("peak_vram_gb", "max"),
        input_tokens_mean=("input_tokens", "mean"),
        input_tokens_std=("input_tokens", "std"),
        output_tokens_mean=("output_tokens", "mean"),
        output_tokens_std=("output_tokens", "std"),
        valid_report_rate=("valid_report", "mean"),
        truncation_rate=("was_truncated", "mean"),
    )
)

operational_path = results_dir / "without_rag_operational_summary.csv"
operational_summary.to_csv(
    operational_path,
    index=False,
    encoding="utf-8",
)

display(operational_summary)
print("Resumo operacional salvo:", operational_path)

## Manifesto de reprodutibilidade

In [ ]:
manifest = {
    "experiment_version": GLOBAL_CONFIG["experiment_version"],
    "condition": "without_rag",
    "models": RUN_MODELS,
    "global_config": GLOBAL_CONFIG,
    "model_profiles": MODEL_PROFILES,
    "environment": {
        "python": platform.python_version(),
        "torch": torch.__version__,
        "transformers": __import__("transformers").__version__,
        "cuda_available": torch.cuda.is_available(),
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    },
    "selected_cases": [
        {
            "sample_id": case.get("sample_id"),
            "predicted_class": case.get("predicted_class"),
            "confidence": float(case.get("confidence")),
        }
        for case in test_cases
    ],
}

manifest_path = results_dir / "manifest_without_rag.json"
manifest_path.write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
print("Manifesto salvo:", manifest_path)

## Exportação dos resultados

Os arquivos principais desta etapa são:

- `generation_<modelo>_without_rag.csv`: checkpoint individual de cada modelo;
- `all_generations_without_rag.csv`: todas as respostas consolidadas;
- `without_rag_operational_summary.csv`: métricas de custo e execução;
- `manifest_without_rag.json`: configurações e ambiente experimental.

Nenhum relatório de referência é solicitado neste notebook.

In [ ]:
import shutil

zip_path = shutil.make_archive(
    "/content/without_rag_results",
    "zip",
    GLOBAL_CONFIG["results_dir"],
)

print("Arquivo compactado:", zip_path)
from google.colab import files
files.download(zip_path)